# 10 - v0.1 Onboarding Journey

This notebook is the runnable version of the v0.1 onboarding probe. It uses only the kernel package surface and the native reasoning path:

1. Define a small schema with primary and secondary identity fields.
2. Write data with `sdk.ref`, `sdk.set`, and `sdk.add`.
3. Read with `sdk.get` and `sdk.run(Query(...))`.
4. Derive and accept a native candidate.
5. Export a kernel audit package.
6. Read the package back with `kernel.audit` and inspect the candidate evidence tree.

No `agent`, `service`, `domains`, PyReason, ProbLog, or external API key is required.

## 0. Imports

In [87]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path

# Works both when launched from repo root and from examples/.
cwd = Path.cwd().resolve()
repo_src = cwd / "src"
if not repo_src.exists() and cwd.name == "examples":
    repo_src = cwd.parent / "src"
if repo_src.exists():
    sys.path.insert(0, str(repo_src))

In [88]:
from kernel.adapters.souffle.package import ExportOptions
from kernel.sdk import Derivation, Entity, Field, Identity, Pred, Query, SDKStore, vars as sdk_vars

## 1. Define the schema

`User.user_id` is the primary identity. `User.locale` is a secondary identity coordinate with a default value. `name` is single-valued, `tag` is multi-valued, and `home` is an entity reference.

In [89]:
class Country(Entity):
    code: str = Identity(primary_key=True)
    name: str = Field(cardinality="single")


class User(Entity):
    user_id: str = Identity(primary_key=True)
    locale: str = Identity(default="zh")
    name: str = Field(cardinality="single")
    tag: str = Field(cardinality="multi")
    home: Country = Field(cardinality="single")

## 2. Create refs and write facts

The v0.1 hardened path lets `sdk.set` and `sdk.add` materialize the entity identity and `<T>:exists` facts through the application write-plan path. The user does not need to start with `sdk.batch`.

In [90]:
sdk = SDKStore([Country, User])

de = sdk.ref(Country, code="DE")
alice = sdk.ref(User, user_id="u1", locale="zh")

sdk.set(Country.name, de, "Germany")
sdk.set(User.name, alice, "Alice")
sdk.set(User.home, alice, de)
sdk.add(User.tag, alice, "admin")
sdk.add(User.tag, alice, "vip")

print(alice)

idref_v1:User:mmrx2ripdh6n4ftyn4wcr7elqbcd4dvknk3db3z5dvv5y2myj2ba


## 3. Read the entity snapshot

In [91]:
snapshot = sdk.get(User, user_id="u1", locale="zh")

print(snapshot.name)
print(sorted(snapshot.tag))
print(snapshot.home)

Alice
['admin', 'vip']
idref_v1:Country:nha7dkfhpr4bajbro7qcluzlc62gsipsx2kjihdpjon6ajtd3lha


## 4. Query with the SDK DSL

In [92]:
with sdk_vars("u", "name") as (u, name):
    q = Query(
        head=[User(u), User.name(value=name)],
        where=[User(u), u.name == name],
    )

rows = sdk.run(q)
rows

[{'u': EntitySnapshot(entity_type='User', ref='idref_v1:User:mmrx2ripdh6n4ftyn4wcr7elqbcd4dvknk3db3z5dvv5y2myj2ba', home='idref_v1:Cou...pjon6ajtd3lha', name='Alice', tag=('admin', 'vip')),
  'name': 'Alice'}]

## 5. Derive and accept a native candidate

This native derivation turns an `admin` tag into a derived `audited` tag. It demonstrates that facts written through `sdk.set/add` are visible to the reasoning path.

In [93]:
with sdk_vars("u", "loc", "derived") as (u, loc, derived):
    derivation = Derivation(
        id="journey.derived_tag",
        version="1.0.0",
        # u.tag == "admin" lowers to the same predicate as Pred("user:tag", u, "admin").
        # The field-sugar form is preferred for schema fields; Pred(...) is the low-level escape hatch.
        where=[User(u), u.locale == loc, u.tag == "admin", derived == "audited"],
        head=User.tag(locale=loc, tag=derived),
    )

candidates = sdk.evaluate(derivation, mode="native")
print(len(candidates))

accepted = sdk.accept(candidates[0], approved_by="journey", note="accept derived audit tag")
accepted

1


AcceptResult(run_id='1a45606e24db4b21b7b3ad3e959dba74', accepted_count=1, skipped_count=0, written_assertions=[{'asrt_id': 'a10335eee9694eb9a89fd26bba1e550d', 'pred_id': 'user:tag', 'key_tuple_digest': 'sha256:9995b3279b00fb933f007ff13bd335ca3ef905cecba8544e8bd234c9ecccb561'}], skipped_reason_counts={}, diagnostics=[], diagnostics_contract_version=1, entity_ref=None, candidate_id='cand_v2:cdffae70466f59345297190a1426e08def277cd6e02936003f2f91ee9be5ea7e', candidate_key='candk_v2:99498175b7da7357b27412857bbb536fac6b851c72ee8ae3fc47054832b11259')

In [94]:
with sdk_vars("u") as (u,):
    derived_query = Query(
        head=User(u),
        where=[User(u), u.tag == "audited"],
    )

derived_rows = sdk.run(derived_query)
derived_rows

[{'u': EntitySnapshot(entity_type='User', ref='idref_v1:User:mmrx2ripdh6n4ftyn4wcr7elqbcd4dvknk3db3z5dvv5y2myj2ba', home='idref_v1:Cou...pjon6ajtd3lha', name='Alice', tag=('admin', 'audited', 'vip'))}]

## 6. Export a kernel audit package

This verifies the kernel audit package boundary. It does not render a static site; presentation is application-owned.

In [95]:
package_dir = Path(tempfile.mkdtemp(prefix="factpy_audit_package_"))
sdk.export_package(package_dir, ExportOptions(package_kind="audit"))

print(package_dir)
print((package_dir / "manifest.json").exists())
print(sorted(p.name for p in package_dir.iterdir()))

/var/folders/05/6btr2vg13b9gvgs3gxt8fw_40000gn/T/factpy_audit_package_uzg1mfrl
True
['audit', 'facts', 'manifest.json', 'outputs', 'policy', 'rules', 'schema']


## 7. Read the audit package back

Export is only half of the delivery story. The same kernel wheel can also read the package back and expose run, candidate, and accept-write ledgers.

In [96]:
from kernel.audit import (
    AuditQuery,
    build_candidate_evidence_tree_dto,
    build_candidate_evidence_tree_narrative_dto,
    build_candidate_evidence_tree_summary_dto,
    load_audit_package,
)
from IPython.display import HTML, display

In [97]:
package = load_audit_package(package_dir)
audit = AuditQuery(package)

runs = audit.list_runs()
accepted_candidates = audit.list_candidates(state="accepted")
accept_writes = audit.list_accept_writes()

runs_view = [
    {
        "run_id": row["run_id"],
        "candidate_count": len(row.get("candidate_ids", [])),
        "decision_count": row.get("decision_count", 0),
        "has_failures": row.get("has_failures", False),
    }
    for row in runs
]

candidates_view = [
    {
        "candidate_id": row["candidate_id"],
        "state": row["state"],
        "pred_id": row["pred_id"],
        "support_kind": row["support_kind"],
    }
    for row in accepted_candidates
]

accept_writes_view = [
    {
        "candidate_id": row["candidate_id"],
        "asrt_id": row["asrt_id"],
        "pred_id": row["pred_id"],
        "approved_by": row.get("approved_by"),
    }
    for row in accept_writes
]

print("runs")
print(runs_view)
print("accepted candidates")
print(candidates_view)
print("accept writes")
print(accept_writes_view)

candidate_id = accepted_candidates[0]["candidate_id"]
candidate_id

runs
[{'run_id': '1a45606e24db4b21b7b3ad3e959dba74', 'candidate_count': 1, 'decision_count': 1, 'has_failures': False}]
accepted candidates
[{'candidate_id': 'cand_v2:cdffae70466f59345297190a1426e08def277cd6e02936003f2f91ee9be5ea7e', 'state': 'accepted', 'pred_id': 'user:tag', 'support_kind': 'native_binding_v1'}]
accept writes
[{'candidate_id': 'cand_v2:cdffae70466f59345297190a1426e08def277cd6e02936003f2f91ee9be5ea7e', 'asrt_id': 'a10335eee9694eb9a89fd26bba1e550d', 'pred_id': 'user:tag', 'approved_by': 'journey'}]


'cand_v2:cdffae70466f59345297190a1426e08def277cd6e02936003f2f91ee9be5ea7e'

## 8. Inspect the candidate evidence tree

Native derivations explain candidates through the candidate evidence tree stored in the audit package. The same package can expose the raw tree, a compact summary, and a narrative DTO.

`EvidenceGraph` is engine-bound: PyReason, ProbLog, and Souffle adapters can materialize graph artifacts from their provenance traces. The native path in this notebook does not emit an `EvidenceGraph`, so the candidate evidence tree below is the primary reader-side explanation.

In [98]:
raw_tree = build_candidate_evidence_tree_dto(audit, candidate_id)
summary_dto = build_candidate_evidence_tree_summary_dto(audit, candidate_id)
narrative_dto = build_candidate_evidence_tree_narrative_dto(audit, candidate_id)

summary = summary_dto["summary"]
narrative = narrative_dto["narrative"]

print(raw_tree["root"]["title"])
print(summary)
print(narrative["headline"])
print(narrative["evidence_lines"])

Candidate cand_v2:cdffae70466f59345297190a1426e08def277cd6e02936003f2f91ee9be5ea7e
{'candidate_id': 'cand_v2:cdffae70466f59345297190a1426e08def277cd6e02936003f2f91ee9be5ea7e', 'support_kind': 'native_binding_v1', 'is_degraded': False, 'root_result_kind': 'fact', 'node_count_by_role': {'structural': 2, 'witness': 6, 'constraint': 1, 'rule_chain': 0, 'terminal': 0, 'degraded': 0}, 'witness_assertion_count': 3, 'rule_ref_count': 0, 'recursive_depth': 0, 'has_unresolved': False, 'has_boundary': False, 'unresolved_reasons': [], 'boundary_reasons': []}
Candidate cand_v2:cdffae70466f59345297190a1426e08def277cd6e02936003f2f91ee9be5ea7e uses support kind native_binding_v1 across 9 tree node(s).
['Witness assertions: 3.', 'Witness nodes: 6; constraint nodes: 1.']


In [99]:
from html import escape


def render_candidate_tree_html(tree):
    def render_node(node):
        title = escape(str(node.get("title") or node.get("node_id") or "node"))
        kind = escape(str(node.get("node_kind") or "node"))
        extras = []
        for key in ("pred_id", "asrt_id", "status", "support_kind"):
            value = node.get(key)
            if value:
                extras.append(f"<code>{escape(key)}={escape(str(value))}</code>")
        extra_html = " ".join(extras)
        children = node.get("children") or []
        child_html = "".join(render_node(child) for child in children)
        nested = f"<ul>{child_html}</ul>" if child_html else ""
        return f"<li><strong>{title}</strong> <code>{kind}</code> {extra_html}{nested}</li>"

    root = tree["root"]
    return f"""
    <div style="font-family: system-ui, -apple-system, Segoe UI, sans-serif; line-height: 1.45;">
      <h4>Candidate evidence tree</h4>
      <p style="color: #555;">Rendered from <code>build_candidate_evidence_tree_dto(...)</code>.</p>
      <ul>{render_node(root)}</ul>
    </div>
    """
display(HTML(render_candidate_tree_html(raw_tree)))